# Day 3 — User journeys and funnel analysis

Objective: map how users move through the product, identify the natural journey flow, and measure where sessions drop out before conversion. This analysis uses the actual cleaned event export and follows the observed GA event taxonomy rather than assuming funnel stages from a generic e-commerce template.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'src').exists(), 'Run this notebook from the repository or notebooks directory.'
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import clean_events, locate_raw_file

raw_path = locate_raw_file(PROJECT_ROOT / 'data' / 'raw')
raw = pd.read_csv(raw_path)
events, _, _ = clean_events(raw)
sessions = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'session_features.csv', parse_dates=['session_start', 'session_end'])
print(f'Loaded {len(events):,} events and {len(sessions):,} sessions.')


## 1) Journey preparation

The source data already contains session boundaries, so the day 3 journey work starts from the cleaned event table and orders each session by `hit_number`. Repeated event names are preserved where they are genuinely part of the behavior, and the analysis uses the real event names supplied by the export.

In [ ]:
ordered = events.sort_values(['session_id', 'hit_number'], kind='stable').copy()
journey_df = (
    ordered.groupby('session_id', sort=False)['event_name']
    .agg(list)
    .reset_index()
    .rename(columns={'event_name': 'event_sequence'})
)
journey_df['journey_length'] = journey_df['event_sequence'].map(len)
journey_df['journey_text'] = journey_df['event_sequence'].map(' -> '.join)
journey_df[['session_id', 'journey_length', 'journey_text']].head(10)


## 2) Event transition analysis

This section examines the moves between consecutive events. The most frequent transitions are not necessarily the most valuable; the purpose here is to understand which actions are connected and which sequences are most common.

In [ ]:
transition_rows = []
for sequence in journey_df['event_sequence']:
    for previous, current in zip(sequence, sequence[1:]):
        transition_rows.append({'from_event': previous, 'to_event': current})

transitions = pd.DataFrame(transition_rows)
transition_counts = (
    transitions.groupby(['from_event', 'to_event'], dropna=False).size().reset_index(name='count')
)
transition_counts['transition_probability'] = transition_counts['count'] / transition_counts['count'].sum()
transition_counts = transition_counts.sort_values('count', ascending=False).reset_index(drop=True)
transition_counts.head(10)


## 3) Common journey patterns

The most common journey is a short browse session that starts on a page and remains on page hits. We only highlight patterns with a meaningful sample size, and conversion rates are reported as observed, not inferred.

In [ ]:
journey_summary = journey_df.merge(
    sessions[['session_id', 'converted', 'entry_page', 'exit_page']], on='session_id', how='left'
)
patterns = (
    journey_summary.groupby('journey_text', dropna=False)
    .agg(sessions=('session_id', 'count'), conversion_rate=('converted', 'mean'))
    .reset_index()
    .sort_values(['sessions', 'conversion_rate'], ascending=[False, False])
    .reset_index(drop=True)
    .query('sessions >= 5')
)
patterns.head(10)


## 4) Funnel definition

The observed data supports a simple product-oriented funnel: product discovery, add to cart, checkout, purchase. These are not assumed labels; they are derived from the actual page-path and event-action patterns present in the GA export. Sessions are counted in order, so a later stage only qualifies if the session has already reached the earlier stage.

In [ ]:
from src.journey_analysis import build_funnel_summary

funnel = build_funnel_summary(events, sessions)
funnel


In [ ]:
funnel_plot = funnel.copy()
funnel_plot['stage'] = funnel_plot['stage'].str.replace('_', ' ').str.title()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(funnel_plot['stage'], funnel_plot['stage_count'], marker='o', color='#2B5D7A')
ax.set_title('Funnel progression by observed stage')
ax.set_xlabel('Funnel stage')
ax.set_ylabel('Sessions reaching stage')
plt.xticks(rotation=20)
fig.tight_layout()
plt.show()


## 5) Drop-off and exit analysis

The largest meaningful drop-off is between product discovery and add-to-cart behavior. The data suggests that many sessions browse the catalog and exit without taking the next explicit purchase action, which is a likely friction or intent gap.

In [ ]:
exit_summary = sessions['exit_page'].value_counts().head(10).reset_index()
exit_summary.columns = ['exit_page', 'sessions']
exit_summary


In [ ]:
device_funnel = (
    sessions.groupby('device_type', dropna=False)
    .agg(sessions=('session_id', 'count'), conversion_rate=('converted', 'mean'))
    .sort_values('sessions', ascending=False)
)
device_funnel


## Key findings

- Observation: most sessions are short browse-only sequences dominated by `PAGE` hits.
- Observation: the highest-volume transition is `PAGE -> PAGE`, followed by `PAGE -> Quickview Click` and `Quickview Click -> PAGE`.
- Observation: the main funnel sharply narrows from product discovery to add-to-cart and then checkout.
- Observation: the most common exit pages are home and product browse pages rather than final purchase pages.
- Interpretation: a large share of sessions shows browsing behavior without an explicit commercial action.
- Business relevance: product detail exposure and add-to-cart friction are the most actionable places to improve conversion intent.

## Limitations

This dataset covers a single day of GA export and does not contain a full customer identity graph. The event taxonomy is page and interaction focused, and conversion is represented by observed transactions rather than a full purchase-intent model. The analysis therefore focuses on behavioral evidence and funnel movement rather than causal claims.